# Chain of Verification workflow

This notebook recreates the article's CoVe workflow:

1. Draft an answer.
2. Generate verification questions from the draft.
3. Answer each verification question independently, without showing the draft.
4. Rewrite the final answer using the verification answers.

Source article: https://moazharu.medium.com/chain-of-verification-the-prompting-pattern-that-makes-llm-answers-check-themselves-f9563ea9e960

## Setup

Install the required packages if needed:

```bash
pip install -r requirements.txt
```

Then create a `.env` file in this same folder with:

```text
OPENAI_API_KEY=your-real-key-here
```

Do not put quotes around the key, and do not share this file.

In [ ]:
import os
from dotenv import load_dotenv
from cove_workflow import run_factored_cove, run_joint_cove, make_openai_call_llm

load_dotenv()
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not found. Create a .env file in this notebook folder and restart the kernel.")

# You can lower max_questions to reduce cost and latency.
MODEL = "gpt-5.5"
MAX_QUESTIONS = 6

call_llm = make_openai_call_llm(model=MODEL)

## Factored CoVe

This is the higher-quality version because the verification-answering calls do not receive the draft.

In [ ]:
question = "Give me a concise biography of Grace Hopper, including major dates and accomplishments."

result = run_factored_cove(
    question=question,
    call_llm=call_llm,
    max_questions=MAX_QUESTIONS,
)

print(result.final)

In [ ]:
print("DRAFT\n", result.draft, "\n", sep="")

print("VERIFICATION QUESTIONS")
for i, q in enumerate(result.verification_questions, 1):
    print(f"{i}. {q}")

print("\nVERIFICATION ANSWERS")
for i, a in enumerate(result.verification_answers, 1):
    print(f"{i}. {a}\n")

print("FINAL\n", result.final, sep="")

## Optional: single-call CoVe

This is cheaper and simpler, but less isolated because the model can see its draft while verifying.

In [ ]:
joint = run_joint_cove(
    question=question,
    call_llm=call_llm,
    max_questions=MAX_QUESTIONS,
)

joint

## Swapping in another model provider

The core workflow only needs a function with this signature:

```python
def call_llm(messages: list[dict[str, str]]) -> str:
    ...
```

Each message has `role` and `content`. Return the assistant text as a string, and `run_factored_cove(...)` will handle the pipeline.